# Data Preparation & Splits

## 1. Configuration

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Ravishi\MSc Final Project\skin-lesion-xai")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

HAM_METADATA = DATA_RAW / "HAM10000_metadata.csv"
HAM_IMAGE_DIRS = [
    DATA_RAW / "HAM10000_images_part_1",
    DATA_RAW / "HAM10000_images_part_2",
]
ISIC_TEST_IMAGES = DATA_RAW / "ISIC2018_Task3_Test_Images"
ISIC_TEST_GT = DATA_RAW / "ISIC2018_Task3_Test_GroundTruth.csv"

SPLIT_TRAIN = DATA_PROCESSED / "train.csv"
SPLIT_VAL = DATA_PROCESSED / "val.csv"
SPLIT_TEST = DATA_PROCESSED / "test_isic2018.csv"

POSITIVE_CLASS = "mel"        # binary target: melanoma vs rest
VAL_FRACTION = 0.15
RANDOM_SEED = 42

print("Project root:", PROJECT_ROOT)
print("HAM10000 metadata found:", HAM_METADATA.exists())


Project root: D:\Ravishi\MSc Final Project\skin-lesion-xai
HAM10000 metadata found: True


## 2. Load HAM10000 metadata and creating the binary target

`melanoma = 1` where `dx == 'mel'`, else `0`.

In [2]:
import pandas as pd

def resolve_ham_image(image_id):
    for d in HAM_IMAGE_DIRS:
        p = d / f"{image_id}.jpg"
        if p.exists():
            return str(p)
    return None

ham = pd.read_csv(HAM_METADATA)
ham["label"] = (ham["dx"] == POSITIVE_CLASS).astype(int)
ham["image_path"] = ham["image_id"].apply(resolve_ham_image)

missing = ham["image_path"].isna().sum()
if missing:
    print(f"WARNING: {missing} images in metadata not found on disk.")
    ham = ham.dropna(subset=["image_path"]).reset_index(drop=True)

print(f"Loaded {len(ham)} images, {ham['lesion_id'].nunique()} unique lesions.")
print(f"Melanoma: {ham['label'].sum()} ({100*ham['label'].mean():.1f}%)")
ham.head()


Loaded 10015 images, 7470 unique lesions.
Melanoma: 1113 (11.1%)


,lesion_id,image_id,dx,dx_type,age,sex,localization,label,image_path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,0,D:\Ravishi\MSc Final Project\skin-lesion-xai\d...
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,0,D:\Ravishi\MSc Final Project\skin-lesion-xai\d...
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,0,D:\Ravishi\MSc Final Project\skin-lesion-xai\d...
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,0,D:\Ravishi\MSc Final Project\skin-lesion-xai\d...
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,0,D:\Ravishi\MSc Final Project\skin-lesion-xai\d...


## 3. Lesion-level train/val split (leakage prevention)

In [5]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=RANDOM_SEED)
train_idx, val_idx = next(splitter.split(ham, y=ham["label"], groups=ham["lesion_id"]))

train_df = ham.iloc[train_idx].reset_index(drop=True)
val_df = ham.iloc[val_idx].reset_index(drop=True)

# Leakage check

overlap = set(train_df["lesion_id"]) & set(val_df["lesion_id"])
print(f"Shared lesion_ids between train and val: {len(overlap)}")
assert len(overlap) == 0, "LEAKAGE DETECTED — do not proceed until this is zero."
print("Leakage check passed.\n")

def summarise(name, df):
    n = len(df)
    if n == 0:
        print(f"  {name:<8} (empty — images not found on disk)")
        return
    pos = int(df["label"].sum())
    lesions = df["lesion_id"].nunique() if "lesion_id" in df.columns else "n/a"
    print(f"  {name:<8} images={n:<6} lesions={str(lesions):<6} "
          f"melanoma={pos:<5} ({100*pos/n:.1f}%)")

summarise("TRAIN", train_df)
summarise("VAL", val_df)


Shared lesion_ids between train and val: 0
Leakage check passed.

  TRAIN    images=8488   lesions=6349   melanoma=927   (10.9%)
  VAL      images=1527   lesions=1121   melanoma=186   (12.2%)


## 4. Load ISIC2018 Task 3 test set (external evaluation)

Handles both published ground-truth formats: the HAM10000-style long format
(a `dx` column) and the ISIC challenge one-hot format (MEL/NV/BCC/... columns).


In [6]:
def load_isic_test():
    if not ISIC_TEST_GT.exists():
        print(f"ISIC ground truth not found at {ISIC_TEST_GT} — skipping test set.")
        return None

    gt = pd.read_csv(ISIC_TEST_GT)
    lower = {c.lower().strip(): c for c in gt.columns}

    if "dx" in lower:
        img_col = lower.get("image_id") or lower.get("image") or gt.columns[0]
        out = pd.DataFrame({
            "image_id": gt[img_col].astype(str),
            "dx": gt[lower["dx"]].astype(str).str.strip().str.lower(),
        })
        if "lesion_id" in lower:
            out["lesion_id"] = gt[lower["lesion_id"]].astype(str)
        fmt = "HAM10000-style (dx column)"
    elif any(c.strip().upper() == "MEL" for c in gt.columns):
        id_col = gt.columns[0]
        class_cols = [c for c in gt.columns if c != id_col]
        out = pd.DataFrame({
            "image_id": gt[id_col].astype(str),
            "dx": gt[class_cols].astype(float).idxmax(axis=1).str.strip().str.lower(),
        })
        fmt = "one-hot (MEL/NV/... columns)"
    else:
        raise ValueError(f"Unrecognised ground-truth format. Columns: {list(gt.columns)}")

    out["label"] = (out["dx"] == POSITIVE_CLASS).astype(int)
    out["image_path"] = out["image_id"].apply(lambda i: str(ISIC_TEST_IMAGES / f"{i}.jpg"))

    exists = out["image_path"].apply(lambda p: Path(p).exists())
    if not exists.all():
        print(f"WARNING: {(~exists).sum()} ISIC test images not found on disk.")
        out = out[exists].reset_index(drop=True)

    print(f"Detected format: {fmt}")
    return out

test_df = load_isic_test()
if test_df is not None:
    summarise("TEST", test_df)


Detected format: HAM10000-style (dx column)
  TEST     (empty — images not found on disk)


## 5. Save splits

These CSVs are read directly by the training notebook (03). Re-run this notebook
whenever you need to regenerate them.


In [ ]:
train_df.to_csv(SPLIT_TRAIN, index=False)
val_df.to_csv(SPLIT_VAL, index=False)
if test_df is not None:
    test_df.to_csv(SPLIT_TEST, index=False)

print(f"Saved to {DATA_PROCESSED}")

Saved to D:\Ravishi\MSc Final Project\skin-lesion-xai\data\processed

NOTE: ISIC2018 shares source institutions with HAM10000, so it is a
held-out test set rather than a fully independent domain. State this
honestly in your methodology.
